<a href="https://colab.research.google.com/github/ahmedfathallahedu/mlintern/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfathallahedu/mlintern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Ranking / Scoring.
Why: The goal is to prioritize content for editors to fix. We are not just classifying content as "good" or "bad"; we need a ranked list so an editor knows exactly which piece of content provides the highest potential for recovery if fixed first. This fits the "Which ones first?" task pattern, where we assign a priority score to help streamline decision-making.



In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/ahmedfathallahedu/mlintern/refs/heads/main/data/raw/content_refresh_anonymized.csv')
print(f"Total rows: {len(df)}")
print(df.head())


Total rows: 30000
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Prediction Target: We will predict the future trend direction of content items, specifically identifying those in a "down" trend.
Source of Label: We will derive our target from the existing trend_direction column. We will create a binary label where "down" trends are marked as 1 and others as 0. Crucially, since trend_direction is calculated from trend_pct, we will use trend_direction to build our label but will never include trend_direction or trend_pct as features in our model to avoid "label leakage" (learning the rule instead of the data).

In [ ]:
# Print column names each on a new line
print("Available columns:")
for col in df.columns:
    print(col)

Available columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


In [ ]:
# Create the binary target and verify
# We identify 'down' trends as the positive class (1)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Define target and forbidden features
target_col = 'is_declining'
forbidden_features = ['trend_direction', 'trend_pct']

# Display distribution to understand the task balance
print(f"Target column '{target_col}' created successfully.")
print("\nTarget class distribution:")
print(df[target_col].value_counts())

# Verification of forbidden features not being used
print(f"\nEnsure these are excluded from model training: {forbidden_features}")

Target column 'is_declining' created successfully.

Target class distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Ensure these are excluded from model training: ['trend_direction', 'trend_pct']


## 3. Success metric

*One metric you can defend. What number means 'good'?*

We will use Precision@K as our primary success metric. A "good" number is defined by the precision score we achieve on the top-K items predicted to decline, compared to a random baseline. Since our goal is to help editors prioritize content for recovery, we need to ensure that the content flagged as "declining" is truly in need of attention, minimizing false alarms.

In [ ]:
# Calculate the baseline conversion rate (percentage of content currently declining)
baseline = df['is_declining'].mean()
print(f"Baseline (current declining rate): {baseline:.2%}")


Baseline (current declining rate): 54.21%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one row per content item per client. This allows us to track performance metrics, such as ctr and engagement_rate, which are specific to a content item's history.

In [ ]:
# Show the dataframe structure and confirm one row per content item
print(f"Dataframe shape: {df.shape}")
print(df[['content_id', 'client_id', 'is_declining']].head())


Dataframe shape: (30000, 45)
             content_id          client_id  is_declining
0  content_304f48230142  client_f369cb89fc             1
1  content_a1fb4e703a9e  client_4e07408562             1
2  content_9aa793d4d895  client_7f2253d7e2             1
3  content_331d6c4de07b  client_19581e27de             0
4  content_d99b7a2d90ca  client_3fdba35f04             1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule (like an if-statement) fails here because declining content is influenced by a complex, non-linear interaction between many variables (e.g., avg_position, scroll_rate, content_type, and competition_level). These variables do not have constant thresholds for decline. Machine learning captures these subtle, multidimensional patterns that a simple hard-coded rule would miss.

In [ ]:
# Check for complex correlations that a simple rule might miss
# Looking at the correlation between engagement and position for declining vs non-declining
correlation_check = df.groupby('is_declining')[['avg_position', 'engagement_rate']].corr()
print(correlation_check)

                              avg_position  engagement_rate
is_declining                                               
0            avg_position         1.000000         0.003075
             engagement_rate      0.003075         1.000000
1            avg_position         1.000000        -0.044941
             engagement_rate     -0.044941         1.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.